# Preprocessing Notebook
### Sentiment / Emotion Classification of YouTube Comments on IHSG

**Flow:**
1. Global Configuration – tweak these variables to match your dataset
2. Load labeled data from the `datasets/` folder
3. Text preprocessing (cleaning, normalisation, stemming)
4. Train/test split & label encoding
5. Save processed arrays to `datasets/processed/` for use in `notebooks/model.ipynb`

## 1. Global Configuration

**Edit this cell** to match your dataset before running anything else.

| Variable | Description |
|---|---|
| `DATASET_DIR` | Folder that contains your labeled CSV files |
| `LABELED_CSV` | Path to the labeled CSV (relative to this notebook or absolute) |
| `TEXT_COLUMN` | Name of the column containing the raw comment text |
| `LABEL_COLUMN` | Name of the column containing the emotion label |
| `ID_COLUMN` | Name of the unique-ID column (set to `None` if absent) |
| `EMOTION_LABELS` | Ordered list of valid label strings (defines class order) |
| `USE_STEMMING` | Toggle Sastrawi stemming (slower but more accurate) |
| `SEED` | Random seed for reproducibility |
| `TEST_SIZE` | Fraction of data held out for testing |
| `N_FOLDS` | Number of stratified k-fold splits for cross-validation |
| `PROCESSED_DIR` | Where to save the processed numpy arrays |
| `MODELS_DIR` | Where `model.ipynb` will save trained models |

In [ ]:
import os

# ============================================================
#  GLOBAL VARIABLES  -  edit these to match your dataset
# ============================================================

# --- Dataset location ---
DATASET_DIR  = "datasets"           # folder that holds your CSV files
LABELED_CSV  = os.path.join(DATASET_DIR, "comments_labeled.csv")  # adjust filename as needed

# --- Column names inside the CSV ---
TEXT_COLUMN  = "text"               # column with raw comment text
LABEL_COLUMN = "label"              # column with emotion label
ID_COLUMN    = "comment_id"         # unique-ID column; set None if absent

# --- Emotion classes (order determines numeric encoding) ---
EMOTION_LABELS = [
    "angry",
    "sad",
    "scared",
    "happy",
    "surprised",
    "neutral",
]

# --- Preprocessing knobs ---
USE_STEMMING = True    # Sastrawi stemming (set False for faster iteration)

# --- Split / cross-val ---
SEED      = 42
TEST_SIZE = 0.20
N_FOLDS   = 5

# --- Output folders ---
PROCESSED_DIR = os.path.join(DATASET_DIR, "processed")
MODELS_DIR    = "models"

# Create output folders if they don't exist
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("Configuration loaded.")
print(f"  Labeled CSV  : {LABELED_CSV}")
print(f"  Text column  : {TEXT_COLUMN}")
print(f"  Label column : {LABEL_COLUMN}")
print(f"  Emotions     : {EMOTION_LABELS}")
print(f"  Stemming     : {USE_STEMMING}")
print(f"  Seed         : {SEED}  |  Test size: {TEST_SIZE}  |  Folds: {N_FOLDS}")
print(f"  Processed dir: {PROCESSED_DIR}")
print(f"  Models dir   : {MODELS_DIR}")

## 2. Load Labeled Data

In [ ]:
import re
import pandas as pd

def unescape_newlines(t):
    """Convert literal '\\n' sequences back to spaces."""
    return re.sub(r"(?:\\n)+", " ", str(t)).strip()

df_lab = pd.read_csv(LABELED_CSV, encoding="utf-8-sig")
print(f"Loaded {len(df_lab):,} rows from '{LABELED_CSV}'")
print(f"Columns: {list(df_lab.columns)}")

df_lab[LABEL_COLUMN] = df_lab[LABEL_COLUMN].astype(str).str.strip().str.lower()

bad = df_lab.loc[~df_lab[LABEL_COLUMN].isin(EMOTION_LABELS), LABEL_COLUMN]
if len(bad):
    print(f"[!] Dropping {len(bad)} rows with unknown labels: {sorted(set(bad))}")
df_lab = df_lab[df_lab[LABEL_COLUMN].isin(EMOTION_LABELS)]

df_lab[TEXT_COLUMN] = df_lab[TEXT_COLUMN].apply(unescape_newlines)
df_lab = df_lab[df_lab[TEXT_COLUMN] != ""]
if ID_COLUMN and ID_COLUMN in df_lab.columns:
    df_lab = df_lab.drop_duplicates(subset=ID_COLUMN)
df_lab = df_lab.reset_index(drop=True)

print(f"\nUsable labeled comments: {len(df_lab):,}")
print(df_lab[LABEL_COLUMN].value_counts().reindex(EMOTION_LABELS))
df_lab[[TEXT_COLUMN, LABEL_COLUMN]].head()

## 3. Text Preprocessing

In [ ]:
import emoji
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

SLANG = {
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak",
    "ngga": "tidak", "tdk": "tidak", "g": "tidak",
    "yg": "yang", "dgn": "dengan", "utk": "untuk", "krn": "karena",
    "bgt": "banget", "sm": "sama",
    "udh": "sudah", "udah": "sudah", "blm": "belum", "jgn": "jangan",
    "tp": "tapi", "klo": "kalau", "kalo": "kalau",
    "gw": "saya", "gue": "saya", "lu": "kamu", "lo": "kamu",
    "emg": "memang", "bs": "bisa",
    "cuan": "untung", "boncos": "rugi", "nyangkut": "rugi",
    "gorengan": "manipulasi",
    "ihsg": "ihsg", "ara": "naik", "arb": "turun",
}

stop_words = set(StopWordRemoverFactory().get_stop_words())
stop_words -= {"tidak", "bukan", "belum", "jangan", "tanpa", "kurang"}

stemmer = StemmerFactory().create_stemmer() if USE_STEMMING else None


def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"@\w+", " ", t)
    t = re.sub(r"#(\w+)", r"\1", t)
    try:
        t = emoji.demojize(t, delimiters=(" ", " "), language="id")
    except Exception:
        t = emoji.replace_emoji(t, replace=" ")
    t = t.replace("_", " ")
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    toks = [SLANG.get(w, w) for w in t.split()]
    toks = [w for w in toks if w not in stop_words and len(w) > 1]
    if stemmer:
        toks = [stemmer.stem(w) for w in toks]
    return " ".join(toks)


print("Cleaning text (this may take a while with stemming enabled)...")
df = df_lab.copy()
df["clean"] = df[TEXT_COLUMN].apply(clean_text)
df = df[df["clean"].str.strip() != ""].reset_index(drop=True)

print(f"Samples after preprocessing: {len(df):,}")
df[[TEXT_COLUMN, "clean", LABEL_COLUMN]].head(10)

## 4. Train / Test Split & Label Encoding

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(EMOTION_LABELS)

df = df[df[LABEL_COLUMN].isin(EMOTION_LABELS)].copy()
y = le.transform(df[LABEL_COLUMN])
K = len(le.classes_)

print("Classes:", list(le.classes_))

min_class_count = df[LABEL_COLUMN].value_counts().min()
assert min_class_count >= max(N_FOLDS, 5), (
    f"A class has only {min_class_count} samples – too few for {N_FOLDS}-fold CV."
)

X_train, X_test, y_train, y_test = train_test_split(
    df["clean"].values, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=SEED,
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

split_report = pd.DataFrame({
    "total": np.bincount(y, minlength=K),
    "train": np.bincount(y_train, minlength=K),
    "test":  np.bincount(y_test,  minlength=K),
}, index=le.classes_)
split_report["test_share"] = (split_report["test"] / split_report["total"]).round(3)
split_report["train_%"]   = (split_report["train"] / split_report["train"].sum() * 100).round(1)
split_report["test_%"]    = (split_report["test"]  / split_report["test"].sum()  * 100).round(1)
print(split_report)

assert (split_report["test_share"] - TEST_SIZE).abs().max() < 0.05
assert split_report["test"].min() >= 5

## 5. Save Processed Data

In [ ]:
import json

np.save(os.path.join(PROCESSED_DIR, "X_train.npy"), X_train)
np.save(os.path.join(PROCESSED_DIR, "X_test.npy"),  X_test)
np.save(os.path.join(PROCESSED_DIR, "y_train.npy"), y_train)
np.save(os.path.join(PROCESSED_DIR, "y_test.npy"),  y_test)
np.save(os.path.join(PROCESSED_DIR, "label_encoder_classes.npy"), le.classes_)

config = {
    "EMOTION_LABELS": list(le.classes_),
    "N_FOLDS": N_FOLDS,
    "SEED": SEED,
    "K": K,
    "MODELS_DIR": MODELS_DIR,
    "PROCESSED_DIR": PROCESSED_DIR,
}
with open(os.path.join(PROCESSED_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("Saved to", PROCESSED_DIR)
print("  X_train:", X_train.shape)
print("  X_test :", X_test.shape)
print("  y_train:", y_train.shape)
print("  y_test :", y_test.shape)
print("  Classes:", list(le.classes_))
print("\nAll done! Run model.ipynb next.")